# Python程序设计（项目实战） · 工单P6《故障诊断模型与部署》
**第8周 | 2课时(80分钟) | 理实一体**  | 教师参考版

| 层 | 任务定位 | 达标要求 |
|----|---------|---------|
| **A 师傅** | 进阶拓展 | 模型对比报告：LR/RF/GBM三模型F1+耗时+可解释性三维评价；示范讲解 |
| **B 组长** | 标准达成 | 对比+调参记录+接入数据库联动 |
| **C 操作员** | 基础完成 | F1≥0.80并保存 |
| **D 学徒** | 跟做保底 | 跟做示例跑通TODO-1, 程序出结果 |
> 每4周按工单成绩动态升降层（连续2次≥B升一层）

## 一、课前资讯（Predict·预测）
> **预习快问**：过拟合的信号是什么？怎么防？

> **课前检查**：模板仓库已克隆更新；导学单与实施计划已提交平台

## 二、PRIMM五步法（先读后写）
本工单按国际编程教学法 **PRIMM** 组织：
1. **Predict 预测**——先运行下方示例，猜输出再验证；
2. **Run 运行**——跑通环境与数据；
3. **Investigate 研究**——读懂starter每段在做什么；
4. **Modify 修改**——完成TODO改造；
5. **Make 创造**——独立完成工单任务。


In [ ]:
# --- 环境与数据准备(勿改, 运行即可) ---
import os, sys, subprocess
os.chdir(os.path.abspath("../"))          # 切到课程包根目录(data/所在处)
print("工作目录:", os.getcwd())


## 三、任务要求（Make）
**实施步骤**
1. 五步流: 划分→训练→交叉验证→调参→保存
2. 混淆矩阵+报告(F1≥0.80)
3. joblib保存model_v1

**验收标准**（工单《P6》「验收标准」四行原文）：`及格(D)`＝`诊断模型F1≥0.80并保存；推理服务可调通` ｜ `良好(C)`＝`接入数据库联动；延迟记录完整` ｜ `优秀(B)`＝`模型对比+调参记录；服务异常处理(404/参数错)` ｜ `超优(A)`＝`优秀+ONNX导出或版本对比实验；P7综合项目立项书`
> **口径说明（别把两套字母混为一谈）**：上四行是**工单《P6》「验收标准」原文**（及格/良好/优秀/超优），与本课课堂分层（D 跟做保底 / C 基础完成 / B 标准达成 / A 进阶拓展）**不同轴**——课堂 C≈工单及格、课堂 B≈工单良好，工单「优秀」档（如 模型对比+调参记录；服务异常处理(404/参数错)）课堂阶梯不覆盖。工程化项目的 `output/report.txt` 已按工单原文逐行取证：p11/p21/p33/p55 直接以这四行为分层口径，p12/p22/p41/p66 另设「工单验收四档对照」节；本机取不了证的子项逐条点名，不假判。


### Starter 1：`# -*- coding: utf-8 -*-`
（教师版：下方为 TODO 完成态，可直接运行；学生版此处留空待填）

In [ ]:
# -*- coding: utf-8 -*-
"""P6 故障诊断模型与推理服务 —— 框架
P6.1: 复用p22特征(或自动合成) → 训练→评估→保存
P6.2: http.server推理服务 + 预训练模型包(教师提供, data/models/)
运行: python W08_ml_service.py train      # 训练+评估
     python W08_ml_service.py serve       # 启动推理服务(端口8899)
     curl -X POST localhost:8899/predict -d '{"features":[...30个数]}'
"""
import json
import sys

MODEL_FILE = "model_v1.joblib"

### Starter 2：`def train():`
（教师版：下方为 TODO 完成态，可直接运行；学生版此处留空待填）

In [ ]:
def train():
    """P6.1 五步流 教师完成态: 划分→训练→交叉验证→调参→保存 + A层三模型对比"""
    import time
    import joblib
    from sklearn.ensemble import GradientBoostingClassifier as GBM, RandomForestClassifier
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import classification_report, confusion_matrix, f1_score
    from sklearn.model_selection import GridSearchCV, train_test_split

    X, y = make_data()                        # 合成或读P2.2产物
    Xtr_all, Xte, ytr_all, yte = train_test_split(
        X, y, stratify=y, random_state=42, test_size=0.25)
    Xtr, _, ytr, _ = train_test_split(          # 再切一份留作早停/对照, 防泄漏
        Xtr_all, ytr_all, stratify=ytr_all, random_state=42, test_size=0.2)

    # ---- A层 模型对比报告: F1 + 训练耗时 + 可解释性 三维评价 ----
    cands = {"LR": LogisticRegression(max_iter=1000),
             "RF": RandomForestClassifier(random_state=42, n_estimators=200),
             "GBM": GBM(random_state=42, n_estimators=200)}
    rows = []
    for name, m in cands.items():
        t0 = time.perf_counter()
        m.fit(Xtr, ytr)
        cost = (time.perf_counter() - t0) * 1000
        f1 = f1_score(yte, m.predict(Xte), average="macro")
        expl = ("系数直接可读" if name == "LR" else "特征重要度可查" )
        rows.append((name, round(f1, 3), round(cost, 1), expl))
        print(f"{name}: F1={f1:.3f} 训练={cost:.1f}ms 可解释性={expl}")
    best = max(rows, key=lambda r: r[1])
    print("决策矩阵最优:", best[0], "(F1最高且可解释性满足机房排障需求)")

    # ---- 交叉验证 + 网格调参 ----
    gs = GridSearchCV(RandomForestClassifier(random_state=42),
                      {"n_estimators": [50, 100, 200],
                       "max_depth": [4, 6, None]}, cv=5, n_jobs=-1)
    gs.fit(Xtr, ytr)
    print("最佳参数:", gs.best_params_, "CV F1=", round(gs.best_score_, 3))
    yhat = gs.predict(Xte)
    print("混淆矩阵:\n", confusion_matrix(yte, yhat))
    print(classification_report(yte, yhat, target_names=["正常", "磨损"]))
    assert f1_score(yte, yhat, average="macro") >= 0.80, "C层验收线 F1≥0.80 未达标"

    joblib.dump(gs.best_estimator_, MODEL_FILE, compress=3)
    print("已保存", MODEL_FILE, gs.best_params_)
    return gs.best_estimator_

### Starter 3：`def make_data(n=400):`
（教师版：下方为 TODO 完成态，可直接运行；学生版此处留空待填）

In [ ]:
def make_data(n=400):
    """合成30维特征(0正常/1磨损): 磨损类时域/频域特征整体抬升; 可用P2.2产物替换"""
    import numpy as np
    rng = np.random.RandomState(7)
    X = rng.normal(0, 1, (n, 30))
    y = rng.binomial(1, 0.5, n)                 # 0正常 1磨损
    X[y == 1, 2:8] += rng.uniform(0.8, 1.6)     # 磨损样本第3-8维抬升
    return X, y

### Starter 4：`def serve():`
（教师版：下方为 TODO 完成态，可直接运行；学生版此处留空待填）

In [ ]:
def serve(forever=True, host="0.0.0.0", port=8899):
    """P6.2 最小推理服务(方案A,教学); 生产级见文末方案B: Litestar+Granian(写决策矩阵)"""
    import os as _os
    import joblib
    from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
    if not _os.path.exists(MODEL_FILE):
        train()                                     # 先训练再服务, notebook 里可单 cell 跑通
    model = joblib.load(MODEL_FILE)

    class H(BaseHTTPRequestHandler):
        def do_POST(self):
            if self.path != "/predict":
                self.send_error(404)
                return
            try:
                body = json.loads(self.rfile.read(
                    int(self.headers["Content-Length"])))
                feats = body["features"]
                assert len(feats) == 30, "features 需 30 维, 收到 %d 维" % len(feats)
                pred = int(model.predict([feats])[0])
                out = {"prediction": pred,
                       "label": "磨损" if pred else "正常",
                       "model": MODEL_FILE}
                code_, payload = 200, out
            except Exception as exc:
                import sys as _sys
                print("[ml-service] 请求错误:", exc, file=_sys.stderr)  # 详情进日志
                code_, payload = 400, {"error": "bad request: 需JSON含30维features"}
            data = json.dumps(payload, ensure_ascii=False).encode()
            self.send_response(code_)
            self.send_header("Content-Type", "application/json")
            self.end_headers()
            self.wfile.write(data)

        def log_message(self, *a):               # 静音, 只保留业务日志
            pass

    print(f"推理服务: http://{host}:{port}/predict (POST 30维特征)")
    if host == "0.0.0.0":
        print("安全提示: 监听0.0.0.0仅限机房局域网教学使用, 切勿暴露到公网")
    srv = ThreadingHTTPServer((host, port), H)
    if forever:
        srv.serve_forever()
    return srv

### Starter 5：`if __name__ == "__main__":`
（教师版：下方为 TODO 完成态，可直接运行；学生版此处留空待填）

In [ ]:
if __name__ == "__main__":
    # notebook 里 sys.argv 是内核启动参数, 不能用它取模式: 先判定再回退
    known = [a for a in sys.argv[1:] if a in ("train", "serve")]
    MODE = known[0] if known else "train"
    if MODE == "serve":
        serve()                    # 阻塞式, 课堂演示单独 cell 运行
    else:
        train()
    print(f"模式={MODE} (serve 会阻塞端口8899; notebook 默认 train 不阻塞)")

### 服务演示（补）：D层/A层 真服务演示 —— 真发一次预测请求 + 延迟量化

工单 P6 的两条硬指标只有跑起来才算达成：
- **D层·能发请求**：加载 `model_v1.joblib` → `POST /predict` → 看真实返回 JSON；
- **A层·延迟量化对比**：连发 N 次，报 p50/p95/p99 与均值——工单口径写明
  “体感由慢请求决定，验收看 p95 不看均值”，均值会被快请求拉低，所以两者必须一起打印。

本 cell 用 `127.0.0.1` + 随机空闲端口起服务、跑完即关，不占用课堂的 8899，也不阻塞内核。


In [ ]:
import json as _json
import socket
import threading
import time
import urllib.error
import urllib.request


def _free_port():
    """让内核先拿到一个空闲端口, 避免与课堂里已起的服务抢 8899"""
    with socket.socket() as s:
        s.bind(("127.0.0.1", 0))
        return s.getsockname()[1]


FREE_PORT = _free_port()


def call(port, feats):
    body = _json.dumps({"features": feats}).encode()
    req = urllib.request.Request("http://127.0.0.1:%d/predict" % port, data=body,
                                headers={"Content-Type": "application/json"},
                                method="POST")
    with urllib.request.urlopen(req, timeout=5) as r:
        return r.status, _json.loads(r.read().decode())


def serve_demo(n=120):
    """D层: 真发一次请求看返回JSON; A层: p50/p95/p99 与均值量化对比"""
    srv = serve(forever=False, host="127.0.0.1", port=FREE_PORT)
    th = threading.Thread(target=srv.serve_forever, daemon=True)
    th.start()
    try:
        time.sleep(0.2)                                  # 等端口就绪
        ok = [float(i % 7) * 0.3 + 1.0 for i in range(30)]   # 确定性请求体
        status, payload = call(FREE_PORT, ok)
        print("D层体验: POST /predict -> HTTP %s 返回=%s" % (status, payload))
        assert status == 200 and payload["label"] in ("正常", "磨损")

        try:                                             # 异常契约也要真跑一次
            call(FREE_PORT, [1.0, 2.0])
            raise AssertionError("少给特征值应当被拒绝")
        except urllib.error.HTTPError as exc:
            print("异常契约: HTTP %s %s" % (exc.code, exc.read().decode()[:70]))

        lat = []
        for _ in range(n):
            t0 = time.perf_counter()
            call(FREE_PORT, ok)
            lat.append((time.perf_counter() - t0) * 1000)
        lat.sort()
        mean = sum(lat) / len(lat)
        pct = lambda q: lat[min(len(lat) - 1, int(round(q / 100 * (len(lat) - 1))))]
        p50, p95, p99 = pct(50), pct(95), pct(99)
        print("A层延迟量化 (%d次真实往返): 均值%.2fms p50=%.2fms p95=%.2fms p99=%.2fms 最大%.2fms"
              % (n, mean, p50, p95, p99, lat[-1]))
        print("口径说明: 验收看 p95=%.2fms(慢请求决定体感), 均值%.2fms 会低估卡顿感, "
              "p95/均值=%.2fx" % (p95, mean, p95 / mean))
        assert p95 > 0 and len(lat) == n
        return {"mean": mean, "p50": p50, "p95": p95, "p99": p99}
    finally:
        srv.shutdown()
        srv.server_close()
        th.join(timeout=3)
        print("服务已关闭(端口 %s 释放), 内核不阻塞" % FREE_PORT)


LATENCY = serve_demo()


### Starter 6：`# ============ P6.2 方案B：Litestar + Granian（生产级`
（教师版：下方为 TODO 完成态，可直接运行；学生版此处留空待填）

In [ ]:
# ============ P6.2 方案B：Litestar + Granian（生产级，全异步高吞吐） ============
def serve_litestar():
    """对比方案A(http.server)：类型安全/异步并发/OpenAPI文档开箱即用
    启动: python -m granian --interface asgi "W08_ml_service:app" --port 8898
    B层任务: 用 ab -n 500 -c 20 分别压两方案，QPS对比写入工单决策矩阵
    """
    try:
        from litestar import Litestar, get, post
    except ImportError:                     # 离线降级: 本机未装 litestar 时不影响 notebook 跑通
        print("litestar 未安装: pip install litestar granian 后重跑本 cell (方案B为对比教学项)")
        return None

    _cache = {}

    def _clf():
        import joblib
        if "m" not in _cache:                       # 懒加载：先train再serve也不迟
            _cache["m"] = joblib.load(MODEL_FILE)
        return _cache["m"]

    @get("/health")
    async def health() -> dict:
        return {"ok": True, "svc": "p66-litestar"}

    @post("/predict")
    async def predict(data: dict) -> dict:
        import numpy as np
        x = np.array(data["features"], dtype=float).reshape(1, -1)
        p = _clf().predict_proba(x)[0].tolist()
        return {"label": int(p[1] > 0.5), "proba_wear": round(p[1], 3)}

    return Litestar([health, predict], logging_config=None)

### Starter 7：`app = serve_litestar()   # granian入口：W08_ml_se`
（教师版：下方为 TODO 完成态，可直接运行；学生版此处留空待填）

In [ ]:
app = serve_litestar()   # granian入口：W08_ml_service:app (litestar缺失时为None)
print("方案B入口:", "已就绪" if app is not None else "跳过(未装 litestar)")

## 四、提交要求与避坑
- 提交: `学号_姓名_P6.py` 或本notebook(含输出)
- ⚠ 特征列序与训练完全一致：feature_names.json对齐
- ⚠ F1≥0.80及格线，调参记录留痕
- ⚠ 接入P5库：新记录→特征→预测→结果回写诊断字段

### 💡 分层提示（先自己试5分钟再看）
**思路提示**
> - 五步流：`train_test_split(stratify=y)` → `fit` → `GridSearchCV(cv=5)` → `best_estimator_` → `joblib.dump`
> - 别忘 `confusion_matrix` + `classification_report(target_names=["正常","磨损"])`，验收线 `F1(macro)≥0.80`
> - 防过拟合信号：训练集 F1≈1 而测试集明显低 → 降 `max_depth`、加 `n_estimators` 平均、保持 stratify
> - A层：LR / RF / GBM 三模型同一测试集对比，记 F1 + 训练耗时(ms) + 可解释性(系数 vs 特征重要度)
> - B层：调参记录写进工单；预测结果回写 P5 库的诊断字段
> - 服务：`http.server` 方案A 够用教学；方案B(Litestar+Granian) 做 QPS 对比，未装依赖时本 cell 自动跳过

| 层 | 这样做 |
|----|--------|
| **D 学徒** | 顺序跑 `make_data()` → `train()` → 看到 F1 与混淆矩阵即达标 |
| **C 操作员** | 独立补全 `train()`，F1≥0.80 且 `model_v1.joblib` 落盘 |
| **B 组长** | 补调参记录 + 与 P5 数据仓库联动（新记录→特征→预测→回写） |
| **A 师傅** | 三模型 F1/耗时/可解释性三维对比表 + 方案A/B QPS 压测对比 |

**报错自查三板斧**：①读最后一行错误信息 → ②检查特征维度是否 30 维、列序是否与训练一致 → ③还不行问同层同学或教师
**自检**：对照上方“验收标准”四行逐条自查并在工单上签字，未做到的档位如实写「未做」——不得只勾 及格(D) 与 优秀(B+)：`及格(D)`＝`诊断模型F1≥0.80并保存；推理服务可调通` ｜ `良好(C)`＝`接入数据库联动；延迟记录完整` ｜ `优秀(B)`＝`模型对比+调参记录；服务异常处理(404/参数错)` ｜ `超优(A)`＝`优秀+ONNX导出或版本对比实验；P7综合项目立项书`
> **闸门落点**：D层“能发请求”＝上方 `serve_demo()` 的真实 `POST /predict` 返回；
> A层“延迟量化对比”＝同 cell 打印的 `均值/p50/p95/p99`，写进工单决策矩阵。

---

## 教师参考答案（来源：`教师参考/solutions.py` · `p66()`）

> **发放纪律：本附录勿随学生版下发。** 课堂按层放提示（D层演示照做/C层只给思路行/A层直接做）。

学生版 TODO 位置 → 下方参考实现的对应关系：

| 学生版 TODO 位置 | 教师完成态所在 cell | 关键实现 |
|---|---|---|
| Starter 2 `train()` | Starter 2 | 分层划分 + LR/RF/GBM 三维对比 + `GridSearchCV` + 混淆矩阵/报告 + F1≥0.80 断言 + joblib 落盘 |
| Starter 3 `make_data()` | Starter 3 | 固定种子合成 30 维，磨损类抬升第 3-8 维（可换 P2.2 产物） |
| Starter 4 `serve()` | Starter 4 | 模型缺失先训练；`forever` 开关便于 notebook 非阻塞演示；请求异常统一 400 |
| Starter 6/7 方案B | Starter 6/7 | `litestar` 未装时优雅跳过，装上即得异步 OpenAPI 服务 |
| A 层三模型对比 | Starter 2 输出行 + 下方 `p66()` | F1/耗时/可解释性三维决策矩阵 |
| B 层 调参留痕+DB联动 | `p66()` | `best_params_` 打印 + 回写字段示例 |

In [ ]:
def p66():
    """教师参考答案: 三模型对比 + 调参 + 混淆矩阵 + 断言F1达标(可直接运行)"""
    import time
    import numpy as np
    from sklearn.ensemble import GradientBoostingClassifier as GBM, RandomForestClassifier
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import classification_report, confusion_matrix, f1_score
    from sklearn.model_selection import GridSearchCV, train_test_split

    rng = np.random.RandomState(7)
    X = rng.normal(0, 1, (400, 30))
    y = rng.binomial(1, .5, 400)
    X[y == 1, 2:8] += rng.uniform(.8, 1.6)
    Xtr, Xte, ytr, yte = train_test_split(X, y, stratify=y, random_state=42)

    rows = []
    for name, m in [("LR", LogisticRegression(max_iter=1000)),
                    ("RF", RandomForestClassifier(random_state=42, n_estimators=200)),
                    ("GBM", GBM(random_state=42, n_estimators=200))]:
        t0 = time.perf_counter()
        m.fit(Xtr, ytr)
        rows.append((name, round(f1_score(yte, m.predict(Xte), average="macro"), 3),
                     round((time.perf_counter() - t0) * 1000, 1)))
    print("A层对比(F1, 耗时ms):", rows)

    gs = GridSearchCV(RandomForestClassifier(random_state=42),
                      {"max_depth": [4, 6, None], "n_estimators": [50, 100]}, cv=5, n_jobs=-1)
    gs.fit(Xtr, ytr)
    yhat = gs.predict(Xte)
    f1 = f1_score(yte, yhat, average="macro")
    print("调参记录:", gs.best_params_, "CV=%.3f" % gs.best_score_)
    print("混淆矩阵:\n", confusion_matrix(yte, yhat))
    print(classification_report(yte, yhat, target_names=["正常", "磨损"]), end="")
    assert f1 >= 0.80, f"F1={f1:.3f} 未达C层验收线0.80"
    import joblib
    joblib.dump(gs.best_estimator_, MODEL_FILE, compress=3)
    print(f"P6 参考实现校验通过 ✓ F1={f1:.3f}≥0.80, 已保存 {MODEL_FILE}; serve 模式见 Starter 4")


p66()

## 运行与验证

上方参考实现可**直接运行**（合成数据、CPU 离线、结果可复现）。
教师可对照学生版逐 cell 讲解：学生填 TODO → 此处给出完成态。

> 多解法对比见同目录《多解法手册.md》与 `教师参考/solutions_multi*.py`。